# 2021년 영역분류 라벨 지역별 취합

17개 시도별 수작업 라벨을 검증하고 TF-IDF 학습용 데이터로 취합한다. TF-IDF의 `fit`은 정답 라벨이 있는 2021년 자료에만 수행하며, 다른 연도는 이후 예측 대상으로 사용한다.

- 데이터 스냅샷: 2026-07-28 지역별 완료 XLSX 17개
- 분석 범위: 데이터 통합 및 QA
- 모델 학습·평가: 미수행
- 데이터 분할 전략: N/A
- 평가 지표: N/A
- CV fold 수: N/A

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.consolidate_2021_area_labels import consolidate_labels, save_outputs  # noqa: E402

np.random.seed(42)
SOURCE_DIR = PROJECT_ROOT / "data/interim"
INPUT_DIR = SOURCE_DIR / "영역분류_라벨링/2021"
OUTPUT_DIR = INPUT_DIR / "통합"

In [ ]:
combined, qa = consolidate_labels(INPUT_DIR, SOURCE_DIR)
paths = save_outputs(combined, qa, OUTPUT_DIR)
qa

In [ ]:
summary = pd.Series(
    {
        "지역 수": combined["지역"].nunique(),
        "통합 행 수": len(combined),
        "지역·원본행 중복": combined.duplicated(["지역", "원본행"]).sum(),
        "taxonomy 정규화": combined["taxonomy_정규화여부"].sum(),
        "주요내용_정제 결측": combined["주요내용_정제_결측"].sum(),
        "정규화 후 라벨 결측": combined[["대영역", "세부영역"]].isna().sum().sum(),
    }
)
summary

In [ ]:
taxonomy_changes = combined.loc[
    combined["taxonomy_정규화여부"],
    ["지역", "원본행", "세부사업명", "대영역_원본", "대영역", "세부영역"],
]
taxonomy_changes.groupby(["대영역_원본", "대영역"], dropna=False).size().rename("행수")

In [ ]:
assert len(combined) == 7301
assert combined["지역"].nunique() == 17
assert not combined.duplicated(["지역", "원본행"]).any()
assert not combined[["대영역", "세부영역", "세부사업명", "분류텍스트"]].isna().any().any()
assert int(combined["taxonomy_정규화여부"].sum()) == 261
assert int(combined["주요내용_정제_결측"].sum()) == 17
paths